# SugarJepa downstream — train the forecaster on a pretrained CGM encoder

Clones the repo and runs the **real** `train_sugar_jepa2.py` (SugarJepaModel2:
SugarOne's parallel cross-attention over basal/bolus/carbs + the pretrained
CGM-JEPA encoder as a 4th auxiliary + multi-scale self-attention). No
reimplementation — this is the repo's own code and metrics (MAE/RMSE/MARD).

You compare **pretrained-init vs random-init**: if X-CGM-JEPA pretraining helped,
the pretrained run wins. Reuse the same notebook for the ablation encoders
(no-SIGReg, CGM-only) by pointing `JEPA_INIT` at a different `.pt`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Load the repo code from Drive

Uses **your exact local code** (the zip you upload), not the GitHub remote —
so uncommitted work is included. Upload `glucose_forecasting_code.zip` to
`MyDrive/XJepa` first.

In [ ]:
CODE_ZIP = "/content/drive/MyDrive/XJepa/glucose_forecasting_code.zip"
!rm -rf /content/glucose-forecasting && mkdir -p /content/glucose-forecasting
!unzip -qo "$CODE_ZIP" -d /content/glucose-forecasting
%cd /content/glucose-forecasting
# Colab already has torch/numpy/pandas; add the repo's other runtime deps.
!pip -q install polars typer huggingface_hub scipy

## Data + pretrained encoder from Drive

In [ ]:
import os

DRIVE_DIR = "/content/drive/MyDrive/XJepa"
ARCHIVE   = f"{DRIVE_DIR}/loop_ai_ready_joined2.rar"        # uploaded RAR (RAR5)
CSV       = "/content/loop_ai_ready_joined2.csv"            # extracted (local, fast)
JEPA_INIT = f"{DRIVE_DIR}/cgm_encoder_xjepa.pt"            # pretrained CGM encoder from x_jepa

# Extract the loop CSV (unar handles RAR5; plain unrar on Colab often can't).
if not os.path.exists(CSV):
    !apt-get -qq install -y unar >/dev/null
    !unar -f -o /content "$ARCHIVE"
assert os.path.exists(CSV), f"CSV not found: {CSV}"
assert os.path.exists(JEPA_INIT), f"upload the pretrained encoder to {JEPA_INIT}"
print("data:", CSV)
print("encoder:", JEPA_INIT)

## Config

Full data + 30 epochs is slow on Colab; drop `EPOCHS` for a quick signal. The
JEPA encoder shape flags are the pretraining defaults (128 / patch 8 / 96 /
3 layers / 6 heads) — leave them so the checkpoint loads.

In [ ]:
EPOCHS     = 10
PATIENCE   = 3    # early stopping (train_sugar_jepa2.py --patience)
INPUT_STEPS = 128
PATCH      = 8
SEED       = 42

## Run A — pretrained encoder (`--jepa-init`)

In [ ]:
!python -m sugar_jepa.train_sugar_jepa2 \
  --csv "$CSV" --input-steps {INPUT_STEPS} --jepa-patch-size {PATCH} \
  --epochs {EPOCHS} --patience {PATIENCE} --seed {SEED} --device cuda \
  --jepa-init "$JEPA_INIT" \
  --out-dir "{DRIVE_DIR}/runs/pretrained"

## Run B — control, random encoder (`--jepa-init ""`)

In [ ]:
!python -m sugar_jepa.train_sugar_jepa2 \
  --csv "$CSV" --input-steps {INPUT_STEPS} --jepa-patch-size {PATCH} \
  --epochs {EPOCHS} --patience {PATIENCE} --seed {SEED} --device cuda \
  --jepa-init "" \
  --out-dir "{DRIVE_DIR}/runs/random"

## Compare metrics (MAE / RMSE / MARD)

In [ ]:
import glob
import pandas as pd

for name in ["pretrained", "random"]:
    csvs = sorted(glob.glob(f"{DRIVE_DIR}/runs/{name}/**/*metrics*.csv", recursive=True))
    print(f"===== {name} =====")
    if not csvs:
        print("  (no metrics csv found yet)")
    for c in csvs:
        print(c)
        print(pd.read_csv(c).to_string(), "\n")